### [ ANN 학습진행 ]
- 

In [16]:
import torch
import torch.nn as nn # 인공신경망 관련 모듈
import torch.nn.functional as F #
import torch.optim as optim

In [ ]:
# ----------------------------------------------------------
# [1] 분류 예시 - iris 품종 분류
# - 데이터 : 피쳐 4개 | 타겟 3개
# - 학습 종류 : 지도학습 + 분류
# - 입력층 : 입력 4개 | 출력층 : 출력 3개, 활성 함수 사용 Softmax
# - 은닉층 : 입력 전층 결과값, 출력 알아서..., 활성함수 ReLU
# ----------------------------------------------------------
# 구성 층 순서대로 입력
class IrisModel(nn.Module):
    # 모델 층 구성 요소 생성
    def __init__(self):
        super().__init__()
        self.hd1_layer = nn.Linear(4, 20)
        self.hd2_layer = nn.Linear(20, 30)
        self.hd3_layer = nn.Linear(30, 15)
        self.out_layer = nn.Linear(15, 3)

    # 순전파 진행 메서드
    def forward(self, data):
        out = F.relu(self.hd1_layer(data))          
        out = F.relu(self.hd2_layer(out))
        out = F.relu(self.hd3_layer(out))
        return self.out_layer(out) # 다중 분류, CrossEntroppyLoss 손실함수 적용

In [8]:
import pandas as pd
df = pd.read_csv('../../[8] 머신러닝/Data/Numbers/iris.csv')
df.head()

,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


In [14]:
# 타겟 컬럼 str ==> LabelEncoder 변환
from sklearn.preprocessing import LabelEncoder
target = LabelEncoder().fit_transform(df['variety'])

# 피쳐 타겟 분리
feature = df[df.columns[:-1]]
print(feature.shape, target.shape)

(150, 4) (150,)


In [15]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(feature, target,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=target)

print(f'{x_train.shape}, {y_train.shape}')
print(f'{x_test.shape}, {y_test.shape}')

(120, 4), (120,)
(30, 4), (30,)


In [18]:
# 인스턴스 : 모델, 최적화, 손실함수
# 학습모델 w, b 모두 초기화
model = IrisModel()

# 최적화 인스턴스 : 모든 충의 w, b를 업데이트
adamOP = optim.Adam(model.parameters())

# 손실계산
lossFn = nn.CrossEntropyLoss()

In [20]:
# 학습관련 설정
# 학습 횟수 에프크
epochs = 10

# 한번에 학습할 데이터 개수 : 배치 사이즈
bs = 40

In [22]:
# 처음 ~ 끝까지 1번 학습 => 1 에포크
for epo in range(epochs):
    loss_history = []
    cnt = int(len(x_train)/bs)
    for idx in range(cnt):
        # 학습용 데이터와 정답 추출 => 텐서화
        # -> 데이터 : Linear 클래스의 float32
        # -> 라벨 : CrossEntropyLoss의 Long
        data = torch.FloatTensor(x_train[bs*idx : bs*(idx+1)].values)
        label = torch.LongTensor(y_train[bs*idx : bs*(idx+1)])
        
        # w, b의 grad 속성 초기화
        adamOP.zero_grad()

        # 순전파 학습
        pre_y = model(data)

        # 손실계산
        loss = lossFn(pre_y, label)

        # 역전파
        loss.backward()

        # 업데이트
        adamOP.step()

        # loss값 누적
        loss_history.append(loss.item())

    print(f'[EPOCH-{epo:02}] Loss : {sum(loss_history)/cnt:.3f}')

[EPOCH-00] Loss : 0.999
[EPOCH-01] Loss : 0.985
[EPOCH-02] Loss : 0.969
[EPOCH-03] Loss : 0.954
[EPOCH-04] Loss : 0.937
[EPOCH-05] Loss : 0.919
[EPOCH-06] Loss : 0.901
[EPOCH-07] Loss : 0.881
[EPOCH-08] Loss : 0.861
[EPOCH-09] Loss : 0.841
